## Symbolic computations for quaternions

In [ ]:
import sympy as sp
import numpy as np
import tqdm
import jax.numpy as jnp
from IPython.display import display

In [ ]:
def quat2list(q, remove_zeros=False):
    res = [q.a, q.b, q.c, q.d]
    if remove_zeros:
        res = [elem for elem in res if elem != 0]
    return res

def reduce_quat(q, exprs):
    res = []
    for i in range(4):
        red = sp.reduced(quat2list(q)[i], exprs)[1]
        res.append(sp.simplify(red))
    return res

In [ ]:
q = sp.algebras.Quaternion(*sp.symbols("q_0:4", real=True))
v = sp.algebras.Quaternion(0, *sp.symbols("v_0:3", real=True))

phi = sp.Symbol(r"\phi", real=True)  # roll
theta = sp.Symbol(r"\theta", real=True)  # pitch
psi = sp.Symbol(r"\psi", real=True)  # yaw
yaw = sp.Symbol("yaw", real=True)

c = sp.algebras.Quaternion(*sp.symbols("c_0:4", real=True))  # current
dt = sp.Symbol(r"\Delta t", real=True)

r = sp.algebras.Quaternion(sp.Symbol("r_0"), sp.Symbol("r_1"), 0, 0)  # yaw unit quat
p = sp.algebras.Quaternion(sp.Symbol("p_0"), 0, sp.Symbol("p_2"), 0)  # yaw unit quat
y = sp.algebras.Quaternion(sp.Symbol("y_0"), 0, 0, sp.Symbol("y_3"))  # yaw unit quat
t = sp.algebras.Quaternion(*sp.symbols("t_0:3", real=True), 0)  # tile unit quat

## yaw-tilt decomposition

In [ ]:
yaw_tilt_eqs = quat2list(y * t - q)
yaw_tilt_eqs.append(y.norm()**2 - 1)
yaw_tilt_vars = quat2list(y, remove_zeros=True) + quat2list(t, remove_zeros=True)
yaw_tilt_sol = sp.solve(yaw_tilt_eqs, yaw_tilt_vars)
assert len(yaw_tilt_sol) == 2
yaw_tilt_sol = yaw_tilt_sol[1]  # take the positive solutions
display(*yaw_tilt_sol)

In [ ]:
yaw_tilt_fun = sp.lambdify(quat2list(q), sp.Matrix([*yaw_tilt_sol]), modules=["jax"], cse=True, docstring_limit=None)
print(yaw_tilt_fun.__doc__)

In [ ]:
def yaw_tilt_decomp(q):
    assert q.shape == (4,)
    q_0, q_1, q_2, q_3 = q
    x0 = jnp.sqrt(q_0**2 + q_3**2)
    x1 = x0**(-1.0)
    res = jnp.ravel(jnp.array([[q_0*x1], [q_3*x1], [x0], [x1*(q_0*q_1 + q_2*q_3)], [x1*(q_0*q_2 - q_1*q_3)]]))
    return jnp.array([res[0], 0, 0, res[1]]), jnp.array([res[2], res[3], res[4], 0])

def turn_tilt_decomp(q):
    assert q.shape == (4,)
    yaw, tilt = yaw_tilt_decomp(q)
    return 2.0 * jnp.atan(yaw[3], yaw[0]), tilt[:3]  # last entry is always zero

## invert yaw-tilt decomp

In [ ]:
inv_yt = sp.simplify((y * q).subs({y.a: sp.cos(yaw / 2), y.d: sp.sin(yaw / 2)}))
inv_yt = sp.Matrix([*quat2list(inv_yt)])

In [ ]:
inv_yt_fun = sp.lambdify([yaw] + quat2list(q), inv_yt, modules=["jax"], cse=True, docstring_limit=None)
print(inv_yt_fun.__doc__)

In [ ]:
def inv_yt_jax(yaw, t):
    assert t.shape == (3,)
    t_0, t_1, t_2 = t
    x0 = (1/2)*yaw
    x1 = jnp.cos(x0)
    x2 = jnp.sin(x0)
    return jnp.array([[t_0*x1], [t_1*x1 - t_2*x2], [t_1*x2 + t_2*x1], [t_0*x2]])

## rotation

In [ ]:
quat = q
# quat = t.subs(zip(quat2list(t, remove_zeros=True), yaw_tilt_sol[2:]))
rot_exprs = reduce_quat(quat * v * quat.conjugate(), [quat.norm()**2 - 1])
# rot_exprs = quat2list(sp.expand(quat * v * quat.conjugate()))
display(*rot_exprs)

In [ ]:
vs = quat2list(v)[1:]
rot_mat = sp.Matrix(
    3,
    3,
    lambda i, j: rot_exprs[i + 1].coeff(vs[j])
)
rot_mat

In [ ]:
rot_mat_fun = sp.lambdify(quat2list(t)[:-1], rot_mat, modules=["jax"], cse=True, docstring_limit=None)
print(rot_mat_fun.__doc__)

In [ ]:
def rot_mat_jax(q):
    assert q.shape == (4,)
    q_0, q_1, q_2, q_3 = q
    x0 = q_1**2
    x1 = q_2**2
    x2 = -x1
    x3 = q_0**2
    x4 = q_3**2
    x5 = x3 - x4
    x6 = 2*q_0
    x7 = q_3*x6
    x8 = q_2*x6
    x9 = 2*q_1
    x10 = -x0
    x11 = q_1*x6
    return jnp.array([[x0 + x2 + x5, 2*q_1*q_2 - x7, q_3*x9 + x8], [q_2*x9 + x7, x1 + x10 + x5, 2*q_2*q_3 - x11], [2*q_1*q_3 - x8, 2*q_2*q_3 + x11, x10 + x2 + x3 + x4]])

def tilt_rot_mat_jax(t):
    assert t.shape == (3,)
    t_0, t_1, t_2 = t
    x0 = 2*t_2**2 - 1
    x1 = 2*t_2
    x2 = t_1*x1
    x3 = t_0*x1
    x4 = 2*t_1**2
    x5 = 2*t_0*t_1
    return jnp.array([[-x0, x2, x3], [x2, 1 - x4, -x5], [-x3, x5, -x0 - x4]])


## euler angles

In [ ]:
R_x = sp.Matrix(
    [
        [1, 0, 0],
        [0, sp.cos(phi), -sp.sin(phi)],
        [0, sp.sin(phi), sp.cos(phi)],
    ]
)
R_y = sp.Matrix(
    [
        [sp.cos(theta), 0, sp.sin(theta)],
        [0, 1, 0],
        [-sp.sin(theta), 0, sp.cos(theta)],
    ]
)
R_z = sp.Matrix(
    [
        [sp.cos(psi), -sp.sin(psi), 0],
        [sp.sin(psi), sp.cos(psi), 0],
        [0, 0, 1],
    ]
)
R =  sp.simplify(R_z * R_y * R_x)
R

In [ ]:
# note: some polynomial relation reductions are possible, but the expressions here are really nice

# thetaq = sp.simplify(sp.asin(-rot_mat[2, 0]))
# phiq = sp.simplify(sp.asin(rot_mat[2, 1] / sp.cos(thetaq)))
# psiq = sp.simplify(sp.asin(rot_mat[1, 0] / sp.cos(thetaq)))

# thetaq = sp.simplify(sp.atan2(-rot_mat[2, 0], ))
phiq = sp.simplify(sp.atan2(rot_mat[2, 1], rot_mat[2, 2]))
psiq = sp.simplify(sp.atan2(rot_mat[1, 0], rot_mat[0, 0]))
thetaq = sp.simplify(sp.atan2(-rot_mat[2, 0], rot_mat[2, 1] / sp.sin(phiq)))

display(thetaq, phiq, psiq)

In [ ]:
def half_cos(x):
    # return sp.sqrt((1 + sp.cos(x)) / 2)
    return sp.cos(x / 2)

def half_sin(x):
    # return x / sp.Abs(x) * sp.sqrt((1 - sp.cos(x)) / 2)
    return sp.sin(x / 2)

In [ ]:
re = sp.simplify(sp.Quaternion(half_cos(phiq), half_sin(phiq), 0, 0))
pe = sp.simplify(sp.Quaternion(half_cos(thetaq), 0, half_sin(thetaq), 0))
ye = sp.simplify(sp.Quaternion(half_cos(psiq), 0, 0, half_sin(psiq)))
display(re, pe, ye)
# display(re)

if True:
    for _ in tqdm.tqdm(range(2**8)):
        q_vars = [q.a, q.b, q.c, q.d]
        q_vals = sp.Quaternion(*np.random.uniform(-1, 1, size=4))
        q_vals = quat2list(q_vals / q_vals.norm())

        tmp0 = (ye * pe * re).subs(zip(q_vars, q_vals))
        tmp1 = rot_mat.subs(zip(q_vars, quat2list(tmp0)))
        tmp2 = R.subs(zip([phi, theta, psi], [phiq, thetaq, psiq])).subs(zip(q_vars, quat2list(tmp0)))
        assert np.allclose(np.array(tmp1 - tmp2, dtype=float), 0.0)

In [ ]:
euler_expr = sp.Matrix([
    phiq,  # roll
    thetaq,  # pitch
    psiq  # yaw
])
euler_fun = sp.lambdify(quat2list(quat, remove_zeros=True), euler_expr, modules=["jax"], cse=True, docstring_limit=None)
print(euler_fun.__doc__)

In [ ]:
def tilt2euler(t):
    assert t.shape == (3,)
    t_0, t_1, t_2 = t
    x0 = 2*t_0
    x1 = t_1**2
    x2 = 2*t_2**2 - 1
    x3 = 2*x1 + x2
    return jnp.ravel(jnp.array([[jnp.arctan2(t_1*x0, -x3)], [jnp.arctan2(t_2*x0, jnp.sqrt(4*t_0**2*x1 + x3**2))], [jnp.arctan2(2*t_1*t_2, -x2)]]))

def quat2euler(q):
    assert q.shape == (4,)
    q_0, q_1, q_2, q_3 = q
    x0 = q_0 * q_1
    x1 = q_2 * q_3
    x2 = 2 * q_2**2 - 1
    x3 = 2 * q_1**2 + x2
    x4 = 2 * q_2
    x5 = 2 * q_3
    return jnp.array([jnp.arctan2(2*x0 + 2*x1, -x3), jnp.arctan2(q_0*x4 - q_1*x5, jnp.sqrt(x3**2 + 4*(x0 + x1)**2)), jnp.arctan2(q_0*x5 + q_1*x4, -2*q_3**2 - x2)])

## angular velocity approximation

In [ ]:
# ang_vel = reduce_quat(c.conjugate() * (c - q) / dt, [c.norm()**2 - 1, q.norm()**2 - 1])
ang_vel = quat2list(sp.simplify(c.conjugate() * (c - q) / dt))
ang_vel = ang_vel[1:]
ang_vel = sp.Matrix([*ang_vel])
display(ang_vel)

In [ ]:
ang_vel_fun = sp.lambdify(quat2list(q) + quat2list(c) + [dt], ang_vel, modules=["jax"], cse=True, docstring_limit=None)
print(ang_vel_fun.__doc__)

In [ ]:
def ang_vel_jax(q, c, dt):
    assert q.shape == (4,) and c.shape == (4,)
    q_0, q_1, q_2, q_3 = q
    c_0, c_1, c_2, c_3 = c
    x0 = 1 / dt
    return jnp.ravel(jnp.array([[x0*(-c_0*q_1 + c_1*q_0 + c_2*q_3 - c_3*q_2)], [x0*(-c_0*q_2 - c_1*q_3 + c_2*q_0 + c_3*q_1)], [x0*(-c_0*q_3 + c_1*q_2 - c_2*q_1 + c_3*q_0)]]))
